In [ ]:
````xml
<!-- filepath: m:\unsloth_clean_for_docs_tests\unsloth_test_3_14_26_pi_day\unsloth\studio\GPT_OSS_20B_Training_Debug_Floaty.ipynb -->
<VSCode.Cell language="markdown">
# GPT-OSS-20B QLoRA Training Debug Notebook
## Issue: Loss Collapse to ~0 within 50-300 steps - Discord User "floaty"

**Hardware:** RTX 3090 24GB VRAM, Ubuntu 24.04  
**Model:** unsloth/gpt-oss-20b (4-bit QLoRA)  
**Dataset:** HuggingFaceH4/Multilingual-Thinking

### 🔴 Critical Issue:
- **Initial loss: 9.874** (expected: ~2.79) ← Problem starts BEFORE training!
- Loss collapses to ~1e-5 within 200-300 steps
- Inference produces token loops ("The The The The...")
- Flash Attention 2 shows as "broken" on user's systems

### 📊 Environment Comparison:
| Metric | Floaty's System (BROKEN) | Leo's Colab (WORKING) |
|--------|--------------------------|------------------------|
| Initial Loss | **9.874** ❌ | **2.79** ✅ |
| Flash Attention | "broken - using Xformers 0.0.35" | Xformers = None |
| Torch | 2.10.0+cu128 | Colab default |
| Result | Token loops | Normal generation |

---
</VSCode.Cell>
<VSCode.Cell language="python">
# ═══════════════════════════════════════════════════════════════════════════════
# 🔧 TEST CONFIGURATION - Toggle settings to reproduce different scenarios
# ═══════════════════════════════════════════════════════════════════════════════

# ────────────────────────────────────────────────────────────────────────────────
# Preset Test Modes (Choose ONE - comment out others)
# ────────────────────────────────────────────────────────────────────────────────

# TEST_MODE = "floaty_original"     # His first attempt (wrong config)
TEST_MODE = "floaty_current"      # Current config (still broken on his system)
# TEST_MODE = "leo_working"         # Leo's working Colab config
# TEST_MODE = "official_notebook"   # Official Unsloth notebook
# TEST_MODE = "custom"              # Manual configuration below

# ────────────────────────────────────────────────────────────────────────────────
# Manual Configuration (used when TEST_MODE = "custom")
# ────────────────────────────────────────────────────────────────────────────────

# Dataset & Model
MAX_SEQ_LENGTH = 3072              # Context length (floaty uses 3072 for 24GB card)
LOAD_IN_4BIT = True                # 4-bit quantization

# Training Hyperparameters  
LEARNING_RATE = 2e-4               # Options: 2e-4 (default), 2e-5 (floaty tried), 1e-4
PER_DEVICE_BATCH_SIZE = 1          # Batch size per device
GRADIENT_ACCUMULATION_STEPS = 4    # Effective batch = 1 * 4 = 4
WARMUP_STEPS = 50                  # Warmup period
NUM_TRAIN_EPOCHS = 1               # Full epoch
MAX_STEPS = None                   # Override epochs if set (e.g., 100, 300 for testing)

# GPT-OSS Specific Settings
USE_TRAIN_ON_RESPONSES_ONLY = False   # ⚠️ Should be False for GPT-OSS (uses 2-channel format)
USE_REASONING_EFFORT = False          # ⚠️ Should be False (not in official notebook)

# Attention & Memory
FORCE_ATTN_IMPLEMENTATION = None   # Options: None (auto), "sdpa", "flash_attention_2", "eager"
DISABLE_FLEX_ATTENTION = False     # Set True to test without flex attention
USE_GRADIENT_CHECKPOINTING = "unsloth"  # Options: "unsloth", True, False
USE_BF16 = True                    # ✅ REQUIRED for RTX 3090

# Evaluation
USE_EVAL_DATASET = True            # Enable evaluation during training
EVAL_STEPS = 50                    # Evaluate every N steps
FP16_FULL_EVAL = True              # Use FP16 for evaluation

# Logging & Debugging
LOGGING_STEPS = 1                  # Log every step
SAVE_STEPS = 50                    # Save checkpoint every N steps
OUTPUT_DIR = "outputs_floaty_debug"

# ────────────────────────────────────────────────────────────────────────────────
# Apply Preset Configurations
# ────────────────────────────────────────────────────────────────────────────────

if TEST_MODE == "floaty_original":
    # floaty's first script - had WRONG settings for GPT-OSS
    print("⚠️  Testing FLOATY ORIGINAL (BROKEN CONFIG)")
    USE_TRAIN_ON_RESPONSES_ONLY = True   # ❌ Wrong - masks analysis channel
    USE_REASONING_EFFORT = True           # ❌ Wrong - not in official notebook
    USE_EVAL_DATASET = True
    LEARNING_RATE = 2e-4
    
elif TEST_MODE == "floaty_current":
    # floaty's corrected script - but STILL BROKEN on his hardware
    print("⚠️  Testing FLOATY CURRENT (FIXED CONFIG, STILL BROKEN)")
    USE_TRAIN_ON_RESPONSES_ONLY = False  # ✅ Fixed
    USE_REASONING_EFFORT = False          # ✅ Fixed  
    USE_EVAL_DATASET = True
    LEARNING_RATE = 2e-4
    
elif TEST_MODE == "leo_working":
    # Leo's Colab - WORKS CORRECTLY
    print("✅ Testing LEO WORKING COLAB CONFIG")
    USE_TRAIN_ON_RESPONSES_ONLY = False
    USE_REASONING_EFFORT = False
    USE_EVAL_DATASET = True
    LEARNING_RATE = 2e-4
    
elif TEST_MODE == "official_notebook":
    # Official Unsloth GPT-OSS notebook
    print("✅ Testing OFFICIAL UNSLOTH NOTEBOOK CONFIG")
    USE_TRAIN_ON_RESPONSES_ONLY = False
    USE_REASONING_EFFORT = False
    USE_EVAL_DATASET = True
    LEARNING_RATE = 2e-4
    MAX_STEPS = None

# ────────────────────────────────────────────────────────────────────────────────
# Environment Tweaks (for testing specific hypotheses)
# ────────────────────────────────────────────────────────────────────────────────

if DISABLE_FLEX_ATTENTION:
    import os
    os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
    print("🔧 Flex attention DISABLED via environment variable")

# ────────────────────────────────────────────────────────────────────────────────
# Display Configuration
# ────────────────────────────────────────────────────────────────────────────────

print("\n" + "═" * 70)
print("🧪 TEST CONFIGURATION")
print("═" * 70)
print(f"Mode:                        {TEST_MODE}")
print(f"Max Seq Length:              {MAX_SEQ_LENGTH}")
print(f"Learning Rate:               {LEARNING_RATE}")
print(f"Batch Size (per device):     {PER_DEVICE_BATCH_SIZE}")
print(f"Gradient Accumulation:       {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective Batch Size:        {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Max Steps:                   {MAX_STEPS if MAX_STEPS else 'Full Epoch'}")
print(f"BF16:                        {USE_BF16}")
print("─" * 70)
print("GPT-OSS Specific:")
print(f"  train_on_responses_only:   {USE_TRAIN_ON_RESPONSES_ONLY} {'❌' if USE_TRAIN_ON_RESPONSES_ONLY else '✅'}")
print(f"  reasoning_effort:          {USE_REASONING_EFFORT} {'❌' if USE_REASONING_EFFORT else '✅'}")
print("─" * 70)
print("Attention/Memory:")
print(f"  Force Attention Impl:      {FORCE_ATTN_IMPLEMENTATION if FORCE_ATTN_IMPLEMENTATION else 'Auto'}")
print(f"  Disable Flex Attention:    {DISABLE_FLEX_ATTENTION}")
print(f"  Gradient Checkpointing:    {USE_GRADIENT_CHECKPOINTING}")
print("─" * 70)
print("Evaluation:")
print(f"  Use Eval Dataset:          {USE_EVAL_DATASET}")
print(f"  Eval Steps:                {EVAL_STEPS if USE_EVAL_DATASET else 'N/A'}")
print("═" * 70)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 1: Installation with diagnostics
import os, importlib.util, sys

print("\n" + "═" * 70)
print("📦 ENVIRONMENT SETUP")
print("═" * 70)

# Check if we need to install
needs_install = importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()) or importlib.util.find_spec("unsloth") is None

if needs_install:
    print("Installing dependencies...")
    !pip install --upgrade -qqq uv

    if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
        try:
            import numpy, PIL
            _numpy = f"numpy=={numpy.__version__}"
            _pil = f"pillow=={PIL.__version__}"
        except:
            _numpy = "numpy"
            _pil = "pillow"

        !uv pip install -qqq \
            "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
            "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
            "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
            git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
    elif importlib.util.find_spec("unsloth") is None:
        !uv pip install -qqq unsloth

    !uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

    print("\n⚠️  Installation complete! Please restart runtime: Runtime → Restart session")
    print("Then run from the next cell.")
else:
    print("✅ Dependencies already installed")

# Print version diagnostics
print("\n" + "─" * 70)
print("📊 PACKAGE VERSIONS")
print("─" * 70)
try:
    import torch
    import transformers
    import unsloth
    import trl
    print(f"PyTorch:       {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA Version:   {torch.version.cuda}")
        print(f"GPU:            {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory:     {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"Transformers:  {transformers.__version__}")
    try:
        print(f"Unsloth:       {unsloth.models._utils.__version__}")
    except:
        print(f"Unsloth:       Installed (version unknown)")
    print(f"TRL:           {trl.__version__}")
except Exception as e:
    print(f"❌ Error checking versions: {e}")

# Check for Flash Attention / Xformers
print("\n" + "─" * 70)
print("🔍 ATTENTION IMPLEMENTATION CHECK")
print("─" * 70)
try:
    try:
        import flash_attn
        print(f"✅ Flash Attention 2: {flash_attn.__version__}")
    except ImportError:
        print("❌ Flash Attention 2: Not installed")
    
    try:
        import xformers
        print(f"⚠️  Xformers: {xformers.__version__}")
    except ImportError:
        print("ℹ️  Xformers: Not installed")
except Exception as e:
    print(f"❌ Error checking attention: {e}")

print("═" * 70)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 2: Load model with diagnostics
from unsloth import FastLanguageModel
import torch

print("\n" + "═" * 70)
print("🔧 LOADING MODEL")
print("═" * 70)

dtype = None  # Auto-detect

model_kwargs = {
    "model_name": "unsloth/gpt-oss-20b",
    "dtype": dtype,
    "max_seq_length": MAX_SEQ_LENGTH,
    "load_in_4bit": LOAD_IN_4BIT,
    "full_finetuning": False,
}

# Add attention implementation if specified
if FORCE_ATTN_IMPLEMENTATION:
    model_kwargs["attn_implementation"] = FORCE_ATTN_IMPLEMENTATION
    print(f"🔧 Forcing attention implementation: {FORCE_ATTN_IMPLEMENTATION}")

print(f"Model: {model_kwargs['model_name']}")
print(f"Max Seq Length: {model_kwargs['max_seq_length']}")
print(f"4-bit Quant: {model_kwargs['load_in_4bit']}")
print("─" * 70)
print("Loading model... (watch for Flash Attention warnings)")
print("─" * 70 + "\n")

model, tokenizer = FastLanguageModel.from_pretrained(**model_kwargs)

print("\n" + "═" * 70)
print("✅ MODEL LOADED")  
print("═" * 70)
print(f"Hidden Size:      {model.config.hidden_size}")
print(f"Num Layers:       {model.config.num_hidden_layers}")
print(f"Attention Heads:  {model.config.num_attention_heads}")
print(f"Vocab Size:       {model.config.vocab_size}")

# Check actual attention implementation used
try:
    if hasattr(model.config, '_attn_implementation'):
        actual_attn = model.config._attn_implementation
        print(f"Attention Impl:   {actual_attn}")
        if actual_attn != FORCE_ATTN_IMPLEMENTATION and FORCE_ATTN_IMPLEMENTATION:
            print(f"⚠️  WARNING: Requested {FORCE_ATTN_IMPLEMENTATION} but got {actual_attn}")
except:
    pass

print("═" * 70)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 3: Apply LoRA with diagnostics
print("\n" + "═" * 70)
print("🔧 APPLYING LORA ADAPTER")
print("═" * 70)

lora_config = {
    "r": 8,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    "lora_alpha": 16,
    "lora_dropout": 0,
    "bias": "none",
    "use_gradient_checkpointing": USE_GRADIENT_CHECKPOINTING,
    "random_state": 3407,
    "use_rslora": False,
    "loftq_config": None,
}

for key, value in lora_config.items():
    print(f"  {key:30s} {value}")
print("─" * 70)

model = FastLanguageModel.get_peft_model(model, **lora_config)

# Count parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_pct = 100 * trainable_params / total_params

print("\n✅ LoRA Applied")
print(f"Trainable Params:  {trainable_params:,} ({trainable_pct:.4f}%)")
print(f"Total Params:      {total_params:,}")
print("═" * 70)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 4: Load and prepare dataset
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt, train_on_responses_only

print("\n" + "═" * 70)
print("📚 LOADING DATASET")
print("═" * 70)

dataset = load_dataset("HuggingFaceH4/Multilingual-Thinking", split="train")

print(f"✅ Dataset loaded: {len(dataset)} examples")
print(f"Dataset features: {list(dataset.features.keys())}")
print("─" * 70)

# Standardize format
dataset = standardize_sharegpt(dataset)
print("✅ Dataset standardized to ShareGPT format")
print("─" * 70)

# Apply chat template
print("Applying chat template...")
if USE_REASONING_EFFORT:
    print("⚠️  WARNING: Using reasoning_effort='medium' (not in official notebook!)")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    if USE_REASONING_EFFORT:
        # floaty's original approach - NOT recommended
        texts = [tokenizer.apply_chat_template(
            convo, 
            tokenize=False, 
            add_generation_prompt=False,
            reasoning_effort="medium"  # ⚠️ Non-standard
        ) for convo in convos]
    else:
        # Official approach
        texts = [tokenizer.apply_chat_template(
            convo, 
            tokenize=False, 
            add_generation_prompt=False
        ) for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"✅ Chat template applied: {len(dataset)} examples")
print("─" * 70)

# Show sample
print("Sample formatted text (first 500 chars):")
print(dataset[0]["text"][:500] + "...")
print("═" * 70)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 5: Create train/test split
print("\n" + "═" * 70)
print("✂️  CREATING TRAIN/TEST SPLIT")
print("═" * 70)

new_dataset = dataset.train_test_split(
    test_size=0.01,  # 1% for test
    shuffle=True,
    seed=3407,
)

train_dataset = new_dataset["train"]
eval_dataset = new_dataset["test"] if USE_EVAL_DATASET else None

print(f"Train dataset:  {len(train_dataset)} examples")
if eval_dataset:
    print(f"Eval dataset:   {len(eval_dataset)} examples")
else:
    print(f"Eval dataset:   Disabled")
print("═" * 70)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 6: Setup trainer with diagnostics
from trl import SFTConfig, SFTTrainer

print("\n" + "═" * 70)
print("🎯 CONFIGURING TRAINER")
print("═" * 70)

training_args_dict = {
    "per_device_train_batch_size": PER_DEVICE_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "warmup_steps": WARMUP_STEPS,
    "learning_rate": LEARNING_RATE,
    "logging_steps": LOGGING_STEPS,
    "optim": "adamw_8bit",
    "weight_decay": 0.01,
    "lr_scheduler_type": "linear",
    "seed": 3407,
    "output_dir": OUTPUT_DIR,
    "report_to": "none",
    "bf16": USE_BF16,
    "save_strategy": "steps",
    "save_steps": SAVE_STEPS,
}

# Add epoch or max_steps
if MAX_STEPS:
    training_args_dict["max_steps"] = MAX_STEPS
else:
    training_args_dict["num_train_epochs"] = NUM_TRAIN_EPOCHS

# Add evaluation settings if enabled
if USE_EVAL_DATASET:
    training_args_dict.update({
        "fp16_full_eval": FP16_FULL_EVAL,
        "per_device_eval_batch_size": 1,
        "eval_accumulation_steps": 4,
        "eval_strategy": "steps",
        "eval_steps": EVAL_STEPS,
    })

training_args = SFTConfig(**training_args_dict)

print("Training Configuration:")
for key, value in training_args_dict.items():
    print(f"  {key:30s} {value}")
print("─" * 70)

# Create trainer
print("Creating trainer...")

trainer_kwargs = {
    "model": model,
    "tokenizer": tokenizer,
    "train_dataset": train_dataset,
    "args": training_args,
}

if USE_EVAL_DATASET:
    trainer_kwargs["eval_dataset"] = eval_dataset

trainer = SFTTrainer(**trainer_kwargs)

# Apply train_on_responses_only if configured (NOT recommended for GPT-OSS)
if USE_TRAIN_ON_RESPONSES_ONLY:
    print("\n⚠️  WARNING: Applying train_on_responses_only()")
    print("   This is WRONG for GPT-OSS (breaks two-channel analysis+final format)")
    gpt_oss_kwargs = dict(
        instruction_part="<|start|>user<|message|>", 
        response_part="<|start|>assistant<|channel|>final<|message|>"
    )
    trainer = train_on_responses_only(trainer, **gpt_oss_kwargs)
    print("   Only training on 'final' channel (analysis channel masked)")
else:
    print("\n✅ NOT using train_on_responses_only (correct for GPT-OSS)")
    print("   Training on full two-channel output (analysis + final)")

print("─" * 70)

# Show what the model will actually see
print("\nSample training input (example 5, first 300 chars):")
print(tokenizer.decode(trainer.train_dataset[5]["input_ids"][:300]))
print("...")

if USE_TRAIN_ON_RESPONSES_ONLY:
    print("\nSample MASKED labels (what gets trained on):")
    labels = trainer.train_dataset[5]["labels"][:300]
    masked_text = tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in labels])
    print(masked_text.replace(tokenizer.pad_token, " "))
    print("...")

print("═" * 70)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 7: Training with enhanced loss monitoring
from transformers import TrainerCallback

print("\n" + "═" * 70)
print("🚀 STARTING TRAINING")
print("═" * 70)
print("⚠️  WATCH FOR:")
print("  1. Initial loss (should be ~2.5-3.0, NOT ~9.87)")
print("  2. Loss progression (gradual decrease, NOT rapid collapse)")
print("  3. Gradient norms (stable, NOT spiking)")
print("  4. Evaluation loss (should track training reasonably)")
print("═" * 70)

# Enhanced diagnostic tracking
class LossDiagnostic:
    def __init__(self):
        self.first_loss = None
        self.all_losses = []
        self.all_eval_losses = []
        self.step = 0
        self.warned_high_initial_loss = False
        self.warned_rapid_collapse = False
        
    def check_first_loss(self, loss):
        if self.first_loss is None:
            self.first_loss = loss
            print("\n" + "─" * 70)
            print(f"🔍 FIRST TRAINING LOSS: {loss:.6f}")
            
            if loss > 5.0:
                print("❌ PROBLEM DETECTED: Initial loss is ABNORMALLY HIGH!")
                print(f"   Expected: ~2.5-3.0")
                print(f"   Got:      {loss:.6f}")
                print("   This indicates model/attention initialization issue BEFORE training!")
                self.warned_high_initial_loss = True
            elif loss > 4.0:
                print("⚠️  Initial loss is higher than expected (~2.79)")
                print(f"   Expected: ~2.5-3.0")
                print(f"   Got:      {loss:.6f}")
            else:
                print("✅ Initial loss looks normal (similar to working config)")
            print("─" * 70 + "\n")
    
    def check_rapid_collapse(self):
        if len(self.all_losses) >= 50 and not self.warned_rapid_collapse:
            recent_50 = self.all_losses[-50:]
            if recent_50[0] > 1.0 and recent_50[-1] < 0.01:
                print("\n" + "─" * 70)
                print("❌ RAPID LOSS COLLAPSE DETECTED!")
                print(f"   Loss dropped from {recent_50[0]:.4f} to {recent_50[-1]:.4f} in 50 steps")
                print("   This matches floaty's issue - model is collapsing!")
                print("─" * 70 + "\n")
                self.warned_rapid_collapse = True

loss_tracker = LossDiagnostic()

class DiagnosticCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if 'loss' in logs:
                loss_tracker.check_first_loss(logs['loss'])
                loss_tracker.all_losses.append(logs['loss'])
                loss_tracker.step += 1
                loss_tracker.check_rapid_collapse()
                
                # Print every 10 steps for first 100 steps
                if loss_tracker.step <= 100 and loss_tracker.step % 10 == 0:
                    print(f"  Step {loss_tracker.step:3d} | Loss: {logs['loss']:.6f} | LR: {logs.get('learning_rate', 0):.2e}")
                
            if 'eval_loss' in logs:
                loss_tracker.all_eval_losses.append(logs['eval_loss'])
                eval_loss = logs['eval_loss']
                train_loss = loss_tracker.all_losses[-1] if loss_tracker.all_losses else None
                
                print(f"\n📊 Eval at step {loss_tracker.step}: Loss = {eval_loss:.4f}", end="")
                if train_loss:
                    gap = eval_loss - train_loss
                    print(f" (Train: {train_loss:.4f}, Gap: {gap:+.4f})", end="")
                    if gap > 10:
                        print(" ⚠️  Very large train/eval gap!", end="")
                print("\n")

trainer.add_callback(DiagnosticCallback())

# Start training
print("Training starting...\n")
trainer.train()

print("\n" + "═" * 70)
print("✅ TRAINING COMPLETE")
print("═" * 70)
print(f"First loss:     {loss_tracker.first_loss:.6f}")
print(f"Final loss:     {loss_tracker.all_losses[-1]:.6f}")
print(f"Total steps:    {len(loss_tracker.all_losses)}")
if loss_tracker.all_eval_losses:
    print(f"Final eval loss: {loss_tracker.all_eval_losses[-1]:.4f}")

# Check for issues
print("\n" + "─" * 70)
print("🔍 DIAGNOSTIC SUMMARY:")
if loss_tracker.first_loss and loss_tracker.first_loss > 5.0:
    print("❌ Abnormal initial loss detected (matches floaty's issue)")
elif loss_tracker.warned_rapid_collapse:
    print("❌ Rapid loss collapse detected (matches floaty's issue)")
else:
    print("✅ Training appears normal")
print("═" * 70)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 8: Test inference with loop detection
print("\n" + "═" * 70)
print("🧪 TESTING INFERENCE")
print("═" * 70)

FastLanguageModel.for_inference(model)

# Test prompts
test_prompts = [
    "Why is the sky blue?",
    "What is 3 + 3?",
    "Explain quantum computing in simple terms."
]

for i, test_prompt in enumerate(test_prompts):
    print(f"\n{'─' * 70}")
    print(f"Test {i+1}: {test_prompt}")
    print('─' * 70)
    
    messages = [{"role": "user", "content": test_prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Extract just the response (after the last message marker)
    if "<|message|>" in generated_text:
        response = generated_text.split("<|message|>")[-1].split("<|")[0]
    else:
        response = generated_text[-500:]  # Last 500 chars
    
    print("Response:")
    print(response[:500])
    
    # Check for token looping
    words = response.split()
    if len(words) > 20:
        last_20 = words[-20:]
        unique_last_20 = set(last_20)
        repetition_ratio = len(unique_last_20) / len(last_20)
        
        if repetition_ratio < 0.3:
            print("\n❌ TOKEN LOOPING DETECTED!")
            print(f"   Last 20 words: {' '.join(last_20)}")
            print(f"   Unique ratio: {repetition_ratio:.2%} (normal: >50%)")
            print("   This matches floaty's issue!")
        elif repetition_ratio < 0.5:
            print(f"\n⚠️  High repetition detected (unique ratio: {repetition_ratio:.2%})")
        else:
            print(f"\n✅ No obvious looping (unique ratio: {repetition_ratio:.2%})")

print("\n" + "═" * 70)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 9: Save model
print("\n" + "═" * 70)
print("💾 SAVING MODEL")
print("═" * 70)

output_dir = f"./{OUTPUT_DIR}_final"
output_dir_merged = f"{output_dir}_merged"

print(f"Saving LoRA adapter: {output_dir}")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Saving merged model: {output_dir_merged}")
model.save_pretrained_merged(output_dir_merged, tokenizer, save_method="mxfp4")

print("\n✅ Model saved successfully")
print("═" * 70)
</VSCode.Cell>
<VSCode.Cell language="markdown">
---

# 📋 Post-Run Checklist

Compare your results to this checklist:

## ✅ Expected Normal Behavior (Working Config):
- [ ] Initial loss: **2.5-3.0** range
- [ ] Loss decreases gradually over 200+ steps
- [ ] Evaluation loss tracks training loss (within ~1-3 points)
- [ ] No "Flash Attention broken" warning (or Xformers=None)
- [ ] Generated text is coherent and on-topic
- [ ] Token repetition ratio >50% in last 20 words

## ❌ Problem Indicators (Matches Floaty's Issue):
- [ ] Initial loss: **>5.0** (especially 9.87+) ← **SMOKING GUN**
- [ ] Loss drops to <0.01 within 50-300 steps
- [ ] Eval loss stays high (>10) while train loss drops
- [ ] Flash Attention shows "broken" message
- [ ] Xformers fallback (version 0.0.35)
- [ ] Generated text loops tokens ("The The The...")
- [ ] Token repetition ratio <30%

---

# 🔬 Next Debugging Steps

If you reproduced the issue, try these configurations:

### Test 1: Disable Flex Attention
```python
DISABLE_FLEX_ATTENTION = True
TEST_MODE = "custom"
```

### Test 2: Force SDPA Attention
```python
FORCE_ATTN_IMPLEMENTATION = "sdpa"
TEST_MODE = "custom"
```

### Test 3: Reduce Learning Rate
```python
LEARNING_RATE = 1e-4  # Even lower than 2e-5
TEST_MODE = "custom"
```

### Test 4: Smaller Batch/Context
```python
MAX_SEQ_LENGTH = 2048
GRADIENT_ACCUMULATION_STEPS = 2
TEST_MODE = "custom"
```

### Test 5: Test Original Wrong Config
```python
TEST_MODE = "floaty_original"  # Should definitely break
```

---

# 📊 Data to Share

If reproducing floaty's issue, share:

1. **Initial loss value** from first step
2. **Flash Attention warning** message from cell 1
3. **Loss progression** for first 100 steps
4. **Inference output** showing loops (or lack thereof)
5. **Environment info** from cell 1 (PyTorch, CUDA versions)

---

**Created by:** Leo [UnAI]  
**For:** Discord user "floaty" - GPT-OSS-20B training debug  
**Date:** April 6, 2026
</VSCode.Cell>
````